In [1]:
import pandas as pd

# قراءة الداتا الطبية الكاملة والمُعالجة من ملف الـ Pickle
df = pd.read_pickle("processed_documents.pkl")

# عرض أول 5 أسطر للتأكد من بنية الحقول الجاهزة للبحث
df.head()

,doc_id,original_text,cleaned_text
0,NCT00000102,Title: Congenital Adrenal Hyperplasia: Calcium...,titl congenit adren hyperplasia calcium channe...
1,NCT00000104,Title: Does Lead Burden Alter Neuropsychologic...,titl lead burden alter neuropsycholog develop ...
2,NCT00000105,Title: Vaccination With Tetanus and KLH to Ass...,titl vaccin tetanu klh assess immun respons co...
3,NCT00000106,Title: 41.8 Degree Centigrade Whole Body Hyper...,titl degre centigrad whole bodi hyperthermia t...
4,NCT00000107,Title: Body Water Content in Cyanotic Congenit...,titl bodi water content cyanot congenit heart ...


In [2]:
from collections import defaultdict

inverted_index = defaultdict(set)

for _, row in df.iterrows():
    doc_id = row['doc_id']

    text = str(row['cleaned_text'])
    words = text.split()

    for word in words:
        inverted_index[word].add(doc_id)

print(" تم بناء الفهرس المعكوس بنجاح")
print("عدد الكلمات المميزة:", len(inverted_index))

 تم بناء الفهرس المعكوس بنجاح
عدد الكلمات المميزة: 265318


### print("أول 5 كلمات من الفهرس المعكوس:")

for word, doc_ids in list(inverted_index.items())[:5]:
    print(f"{word} -> {len(doc_ids)} وثيقة")

In [4]:
import math
from collections import Counter

tf_docs = {}

for _, row in df.iterrows():

    doc_id = row['doc_id']

    words = str(row['cleaned_text']).split()

    counts = Counter(words)

    tf_docs[doc_id] = {
        word: math.log(1 + count)
        for word, count in counts.items()
    }

print("تم حساب TF")

تم حساب TF


In [5]:
import math

N = len(df)

df_counts = {
    word: len(doc_ids)
    for word, doc_ids in inverted_index.items()
}

idf = {
    word: math.log(1 + (N / df_t))
    for word, df_t in df_counts.items()
}

print("تم حساب IDF")

تم حساب IDF


In [6]:
tfidf_docs = {}

for doc_id, tf_dict in tf_docs.items():

    tfidf_docs[doc_id] = {
        word: tf_value * idf[word]
        for word, tf_value in tf_dict.items()
    }

print("تم حساب TF-IDF")

تم حساب TF-IDF


In [7]:
first_doc_id = df['doc_id'].iloc[0]

print(f"TF للوثيقة {first_doc_id}:\n")

for word, score in list(tf_docs[first_doc_id].items())[:5]:
    print(f"{word}: {score:.4f}")

TF للوثيقة NCT00000102:

titl: 0.6931
congenit: 1.3863
adren: 1.3863
hyperplasia: 1.3863
calcium: 1.0986


In [8]:
print("أول 5 كلمات مع قيم IDF:\n")

for word, score in list(idf.items())[:5]:
    print(f"{word}: {score:.4f}")

أول 5 كلمات مع قيم IDF:

titl: 0.6931
congenit: 4.9206
adren: 6.0609
hyperplasia: 5.8599
calcium: 4.4898


In [9]:
first_doc_id = df['doc_id'].iloc[0]

print(f"TF-IDF للوثيقة {first_doc_id}:\n")

for word, score in list(tfidf_docs[first_doc_id].items())[:5]:
    print(f"{word}: {score:.4f}")

TF-IDF للوثيقة NCT00000102:

titl: 0.4805
congenit: 6.8213
adren: 8.4022
hyperplasia: 8.1236
calcium: 4.9325


In [25]:
import math
from collections import Counter
import nltk
from nltk.corpus import stopwords
from nltk.stem import PorterStemmer
from nltk.tokenize import word_tokenize

# 1. تأكدي من تحميل الحزم اللغوية وتجهيز تابع التنظيف
nltk.download("punkt", quiet=True)
nltk.download("stopwords", quiet=True)

stemmer = PorterStemmer()
stop_words = set(stopwords.words("english"))


def preprocess_text(text):
    tokens = word_tokenize(text.lower())
    cleaned_tokens = [
        stemmer.stem(word)
        for word in tokens
        if word.isalnum() and word not in stop_words
    ]
    return " ".join(cleaned_tokens)


# 2. حساب أطوال الوثائق والمتوسط (تأكدي أن df و inverted_index معرفين في الخلايا السابقة)
doc_lengths = {}
for _, row in df.iterrows():
    doc_id = row["doc_id"]
    words = str(row["cleaned_text"]).split()
    doc_lengths[doc_id] = len(words)

avg_doc_length = sum(doc_lengths.values()) / len(doc_lengths)
print("متوسط طول الوثائق:", avg_doc_length)

# 3. حساب تكرارات الكلمات والـ IDF الخاص بـ BM25
df_counts = {}
for word, docs in inverted_index.items():
    df_counts[word] = len(docs)
print("عدد الكلمات في الفهرس:", len(df_counts))

N = len(df)
bm25_idf = {}
for term, df_term in df_counts.items():
    bm25_idf[term] = math.log(((N - df_term + 0.5) / (df_term + 0.5)) + 1)
print("👍 تم حساب BM25 IDF وتجهيز المتغيرات بنجاح!")


# 4. تابع البحث السريع بعد تعديله لمنع الـ NameError نهائياً
def bm25_search_fast(query, bm25_idf, avg_doc_length, k1=1.5, b=0.75):
    scores = {}

    # تنظيف الاستعلام وعمل Stemming ليطابق الفهرس المعكوس
    query_cleaned = preprocess_text(query)
    query_terms = query_cleaned.split()

    if not query_terms:
        return []

    candidate_docs = set()
    for term in query_terms:
        if term in inverted_index:
            candidate_docs.update(inverted_index[term])

    # حساب الـ Scores للوثائق المرشحة
    for doc_id in candidate_docs:
        doc_row = df[df["doc_id"] == doc_id].iloc[0]
        words = str(doc_row["cleaned_text"]).split()

        doc_len = len(words)
        term_freqs = Counter(words)
        score = 0

        for term in query_terms:
            if term not in term_freqs:
                continue

            tf = term_freqs[term]
            idf = bm25_idf.get(term, 0)

            numerator = tf * (k1 + 1)
            denominator = tf + k1 * (1 - b + b * (doc_len / avg_doc_length))

            score += idf * (numerator / denominator)

        scores[doc_id] = score

    ranked_docs = sorted(scores.items(), key=lambda x: x[1], reverse=True)
    return ranked_docs


# 5. استدعاء التابع وتمرير المتغيرات المحسوبة له بشكل صريح
results = bm25_search_fast(
    "machine learning", bm25_idf, avg_doc_length, k1=1.5, b=0.75
)

# 6. طباعة المخرجات
print("\nأفضل 10 نتائج:")
if not results:
    print("لا توجد نتائج تطابق هذا البحث في المستندات الحالية.")
else:
    for doc_id, score in results[:10]:
        print(f"Doc ID: {doc_id} | Score: {round(score, 4)}")

متوسط طول الوثائق: 180.5467676660099
عدد الكلمات في الفهرس: 265318
👍 تم حساب BM25 IDF وتجهيز المتغيرات بنجاح!

أفضل 10 نتائج:
Doc ID: NCT04219306 | Score: 16.6332
Doc ID: NCT04423991 | Score: 16.2117
Doc ID: NCT04682756 | Score: 15.607
Doc ID: NCT04527094 | Score: 15.5786
Doc ID: NCT04839198 | Score: 15.5292
Doc ID: NCT04060706 | Score: 15.4318
Doc ID: NCT04828915 | Score: 15.3803
Doc ID: NCT03175302 | Score: 15.358
Doc ID: NCT04849312 | Score: 15.2342
Doc ID: NCT04784351 | Score: 15.1264
